In [154]:
from bs4 import BeautifulSoup
import requests
import json
import typing
import time

In [177]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
}

In [178]:
def get_detailed_info(movie_url: str, debug=False):
    resp = requests.get(movie_url, headers=HEADERS)
    soup = BeautifulSoup(resp.text)

    script = soup.find("script", {"type": "application/ld+json"})
    
    if debug:
        print(f"Soup: {soup}")
        print(f"Script: {script}")

    raw = script.string.strip()
    clean = raw.replace("/* <![CDATA[ */", "").replace("/* ]]> */", "").strip()
    data = json.loads(clean)

    metadata = {
            "posterUrl": data["image"],
            "director": data["director"],
            "productionCompany": data["productionCompany"],
            "genre": data["genre"],
            "countryOfOrigin": data["countryOfOrigin"],
            "aggregateRating": data["aggregateRating"],
        }

    return metadata


In [184]:
def get_page_info(max_pages: int):
    """
    https://letterboxd.com/films/popular/this/week/

    Goes to the link above and retrieves information about the movies there. 

    NOTE: Does not get movie poster URL's. For that, you need to visit the main page of each movie. This will be done in 'get_poster_urls'.
    
    """

    film_dict = {}

    for i in range(max_pages):
        page_url = f"https://letterboxd.com/films/ajax/popular/this/week/page/{i+1}/?esiAllowFilters=true"
        resp = requests.get(page_url, headers=HEADERS)
        soup = BeautifulSoup(resp.text)
                
        data = soup.select("li.posteritem")
        for data_block in data:
            block = data_block.find("div", class_="react-component")

            metadata = {
                "title": block.get("data-item-full-display-name"),
                "slug": block.get("data-item-slug"),
                "film_id": block.get("data-film-id"),
                "details": block.get("data-details-endpoint"),
                "poster_url": block.get("data-poster-url"),
                "page_link": block.get("data-item-link"),
                "rating": data_block.get("data-average-rating")
            }

            print(f"https://letterboxd.com/film/{metadata['slug']}/")
            temp_metadata = get_detailed_info(f"https://letterboxd.com/film/{metadata['slug']}/")

            for key, value in temp_metadata.items():
                metadata[key] = value

            film_dict[f"https://letterboxd.com/film/{metadata['slug']}/"] = metadata # type: ignore
            time.sleep(1)
            break
        time.sleep(1)
    return film_dict


In [185]:
get_page_info(1)

https://letterboxd.com/film/wicked-for-good/


{'https://letterboxd.com/film/wicked-for-good/': {'title': 'Wicked: For Good (2025)',
  'slug': 'wicked-for-good',
  'film_id': '871148',
  'details': '/film/wicked-for-good/json/',
  'poster_url': '/film/wicked-for-good/image-150/',
  'page_link': '/film/wicked-for-good/',
  'rating': '3.59',
  'posterUrl': 'https://a.ltrbxd.com/resized/film-poster/8/7/1/1/4/8/871148-wicked-for-good-0-230-0-345-crop.jpg?v=f1330d6f91',
  'director': [{'@type': 'Person',
    'name': 'Jon M. Chu',
    'sameAs': '/director/jon-m-chu-2/'}],
  'productionCompany': [{'@type': 'Organization',
    'name': 'Universal Pictures',
    'sameAs': '/studio/universal-pictures/'},
   {'@type': 'Organization',
    'name': 'Marc Platt Productions',
    'sameAs': '/studio/marc-platt-productions/'}],
  'genre': ['Adventure', 'Fantasy', 'Romance'],
  'countryOfOrigin': [{'@type': 'Country', 'name': 'USA'}],
  'aggregateRating': {'bestRating': 5,
   'reviewCount': 370364,
   '@type': 'aggregateRating',
   'ratingValue': 3.59

In [ ]:
# get_detailed_info(movie_url="https://letterboxd.com/film/stranger-things-5-the-finale/", debug=True)

Soup: 
<!DOCTYPE html>

<html class="context-client-unknown no-mobile no-js" id="html" lang="en">
<head>
<meta charset="utf-8"/>
<meta content="width=1024" name="viewport"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<meta content="The fall of 1987. Hawkins is scarred by the opening of the Rifts, and our heroes are united by a single goal: find and kill Vecna. But he has vanished — his whereabouts and plans unknown. Complicating their mission, the government has placed the town under military quarantine and intensified its hunt for Eleven, forcing her back into hiding. As the anniversary of Will’s disappearance approaches, so does a heavy, familiar dread. The final battle is looming - and with it, a darkness more powerful and more deadly than anything they’ve faced before. To end this nightmare, they’ll need everyone - the full party - standing together, one last time." name="description"/>
<meta content="video.movie" property="og:type"/>
<meta content="https://let

KeyError: 'aggregateRating'